In [0]:
import joblib
import pandas as pd

# Load saved model
model = joblib.load("/Workspace/Users/aviola6@gatech.edu/enterprise-data-pipeline/enterprise-data-pipeline/Models/churn_model.pkl")

#Load saved pipeline
pipeline = joblib.load("/Workspace/Users/aviola6@gatech.edu/enterprise-data-pipeline/enterprise-data-pipeline/Models/pipeline.pkl")

# Load new customer data
new_customers = pd.read_csv("/Workspace/Users/aviola6@gatech.edu/enterprise-data-pipeline/enterprise-data-pipeline/data/new_customers.csv")

In [0]:
def aggregate_customer_data(df):

    # Ensure correct types
    df["Purchase_Amount"] = pd.to_numeric(df["Purchase_Amount"], errors="coerce")
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df["Purchase_Date"] = pd.to_datetime(df["Purchase_Date"], errors="coerce")
    df["Discount_Applied"] = df["Discount_Applied"].map({"Yes": 1, "No": 0})

    # Aggregate by Customer_ID
    customer_df = df.groupby("Customer_ID").agg({
        "Purchase_Amount": ["mean", "sum", "count"],
        "Discount_Applied": "mean",
        "Rating": "mean",
        "Purchase_Date": ["min", "max"],
        "Repeat_Customer": "max",  # Keep this as the label
        "Category": lambda x: x.mode()[0] if not x.mode().empty else None
    }).reset_index()

    # Flatten column names
    customer_df.columns = ["_".join(col).strip("_") for col in customer_df.columns.values]

    # Create custom features
    customer_df["Purchase_Span_Days"] = (customer_df["Purchase_Date_max"] - customer_df["Purchase_Date_min"]).dt.days
    customer_df["Repeat_Customer_Flag"] = customer_df["Repeat_Customer_max"].map({"Yes": 1, "No": 0})

    # Rename for clarity
    customer_df.rename(columns={
        "Purchase_Amount_mean": "Avg_Purchase_Amount",
        "Purchase_Amount_sum": "Total_Spend",
        "Purchase_Amount_count": "Total_Purchases",
        "Discount_Applied_mean": "Discount_Rate",
        "Rating_mean": "Avg_Rating",
        "Category_<lambda>": "Most_Common_Category"
    }, inplace=True)

    # Drop old columns
    customer_df.drop(columns=["Purchase_Date_min", "Purchase_Date_max", "Repeat_Customer_max"], inplace=True)

    return customer_df

df = aggregate_customer_data(new_customers)   

In [0]:

# Preprocess: must match training pipeline
X_new = pipeline.transform(df)

In [0]:
# Predict churn probabilities
pred_probs = model.predict_proba(X_new)[:, 1]  # probability of returning
pred_labels = model.predict(X_new)

In [0]:
# Combine predictions with customer IDs
new_customers["Predicted_Return"] = pred_labels
new_customers["Return_Probability"] = pred_probs

# Save scored results
new_customers.to_csv("scored_customers.csv", index=False)